# Experiment 2 · Zero-shot foundation models

This experiment evaluates five promptable foundation segmentation models — **SAM (ViT-H)**, **SAM2**, **MedSAM**, **MedSAM-2**, and **SAM3** — in a pure **zero-shot** setting: each model is run **box-prompted with no training or fine-tuning** on the held-out test split of all four thyroid ultrasound datasets (DDTI, TN3K, ThyroidXL, Stanford AIMI).

Each image is prompted with the ground-truth bounding box of the nodule. By default the box is **tight** (`--jitter_frac 0.0`), which is the protocol used for the reported results; pass `--jitter_frac 0.1` to evaluate with ±10% jittered boxes instead. This is a **5 models × 4 datasets = 20-run** grid. Per-image DSC, IoU, precision, recall, and HD95 are written under `results/<model>_<dataset>/`, and `compute_stats.py` aggregates bootstrap 95% CIs and pairwise Wilcoxon tests.

In [ ]:
# Move to the repository root (the directory that contains pyproject.toml) so that
# the `thyroidbench` package is importable and the relative --data_root / --split_dir
# defaults resolve correctly.
import os
from pathlib import Path

cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists():
        os.chdir(candidate)
        break
print("Repository root:", Path.cwd())

In [ ]:
# Check the inputs this experiment needs before doing anything slow. A missing
# dataset here means an unrun (or unplaced) setup step, not a bug in the experiment.
from pathlib import Path

REQUIRED = ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']
GATED = {'thyroidxl', 'stanford_aimi'}

missing = [d for d in REQUIRED
           if not (Path('data/processed') / d / 'images').is_dir()
           or not (Path('data/splits') / f'{d}_test.csv').exists()]
if missing:
    print('Missing preprocessed data or splits for:', ', '.join(missing))
    for d in missing:
        if d in GATED:
            print(f'  {d:14s} access-gated -> place your approved copy first, see '
                  f'00_setup/01_place_gated_datasets.ipynb')
        else:
            print(f'  {d:14s} open -> download it with 00_setup/00_get_open_datasets.ipynb')
    print('Then run 00_setup/02_preprocess.ipynb and 00_setup/03_make_splits.ipynb.')
    print('\nYou can still run this experiment on whichever datasets ARE present '
          'by restricting the --dataset argument below.')
else:
    print('All four datasets are preprocessed and split.')


## Prerequisites

Foundation-model weights are **not** shipped with the repository. Download the published checkpoints and place them under a git-ignored `pretrained_models/` directory at the repository root. Each wrapper in `thyroidbench/models/` documents the exact file it loads:

| Model | Wrapper | Checkpoint path |
|-------|---------|-----------------|
| SAM (ViT-H) | `thyroidbench/models/sam_wrapper.py` | `pretrained_models/sam/sam_vit_h_4b8939.pth` |
| SAM2 | `thyroidbench/models/sam2_wrapper.py` | `pretrained_models/sam2/sam2.1_hiera_large.pt` |
| MedSAM | `thyroidbench/models/medsam_wrapper.py` | `pretrained_models/medsam/medsam_vit_b.pth` |
| MedSAM-2 | `thyroidbench/models/medsam2_wrapper.py` | `pretrained_models/medsam2/MedSAM2_latest.pt` |
| SAM3 | `thyroidbench/models/sam3_wrapper.py` | `pretrained_models/sam3_vanilla/sam3.pt` |

You also need the preprocessed datasets in `data/processed/` and the split CSVs in `data/splits/` (see `00_setup/`). Runs log to Weights & Biases (project `thyroidbench`); set `WANDB_MODE=offline` to disable.

## Run

The cell below sweeps all 20 (model, dataset) combinations. Each call runs zero-shot inference and writes `results/<model>_<dataset>/per_image_results.csv` and `summary_metrics.csv`. This is inference-only — no training — but the ThyroidXL and Stanford AIMI test splits are large, so the full sweep still takes several hours on a single GPU.

In [ ]:
for model in ['sam', 'sam2', 'medsam', 'medsam2', 'sam3']:
    for dataset in ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']:
        !python experiments/exp2_zeroshot/run.py --model {model} --dataset {dataset}

## Results

The per-cell summaries are collated in `results/zeroshot_summary.csv`. The cell below loads it and displays mean DSC / IoU / HD95 pivoted by model and dataset — the numbers behind Table 2. Run `python experiments/exp2_zeroshot/compute_stats.py` to regenerate the bootstrap CIs and pairwise Wilcoxon tables under `results/stats/`.

In [ ]:
import pandas as pd

summary = pd.read_csv('experiments/exp2_zeroshot/results/zeroshot_summary.csv')

cols = ['model', 'dataset', 'n_images', 'n_skipped',
        'dice_mean', 'iou_mean', 'hd95_mean']
view = summary[cols].sort_values(['model', 'dataset']).reset_index(drop=True)
display(view)

# Mean DSC pivoted as in Table 2 (rows = model, columns = dataset)
dsc_table = summary.pivot_table(index='model', columns='dataset', values='dice_mean')
display(dsc_table.round(4))